In [ ]:
# hide
import numpy as np
import pyquist as pq

amen = pq.Audio.from_file("../assets/amen.wav")

In [ ]:
beat = 0.436                                  # one beat of the break, in seconds


def amen_sample(offset: float, duration: float) -> pq.Audio:
    """An instrument that plays one sample of the Amen break."""
    sample = amen.segment(offset=offset, duration=duration)
    t = np.arange(len(sample)) / sample.sample_rate
    T = sample.duration                       # (the last beat may run past the end)
    env = np.interp(t, [0.0, 0.001, T - 0.001, T], [0.0, 1.0, 1.0, 0.0])
    return sample * env[:, np.newaxis]        # 1 ms fades smooth the cut points


# One event per beat, each playing that same beat: the original break
amen_orig = pq.Score([(i * beat, {"offset": i * beat, "duration": beat}) for i in range(16)])
pq.play(amen_orig.render(amen_sample))

In [ ]:
def amen_chop(hits: int, speed: float = 1.0, seed: int | None = None) -> pq.Score:
    """A random re-sequencing of the break's 16 beats, one hit every beat / speed seconds."""
    rng = np.random.default_rng(seed)
    step = beat / speed
    beats = rng.integers(0, 16, size=hits)    # which beat of the break each hit plays
    return pq.Score([(i * step, {"offset": b * beat, "duration": step})
                     for i, b in enumerate(beats)])


chop = amen_chop(32, speed=1.5, seed=0)
pq.play(chop.render(amen_sample))